# Power the has_error=1 stratum at 3B -- is the 7B result model-size-independent?

Mirrors `pilot/06_7b_error_stratum_power.ipynb` exactly, on the 3B model
instead of 7B. This is the one experiment that can actually change the
report's claim about Phase 4, not just add color: 7B's has_error=1 stratum
went from an underpowered point estimate (0.761, 17 wrong) to a confirmed
0.834 (38 wrong) once given 200 more `has_error`=1 items. **3B's own
n=300 run has the same shape but was never extended** -- its point estimate
is 0.836 [0.626, 0.946] on just **8** misgraded items
(`results/scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv`,
report \S7.2). If 3B also confirms near 0.83-0.84 once powered, the
Phase 4 finding is model-size-independent. If it doesn't, 7B's result was
specific to a bigger model's behavior, not a general property of this task.

## Sizing: why N_EXTRA=500, not 200 like the 7B run

3B's `has_error`=1 error rate (8/150 = 5.3%) is roughly **half** 7B's
(17/150 = 11.3%) -- because 3B says "there is an error" far more often (93%
vs 80%), so it gets *more* `has_error`=1 items right by default, leaving
fewer wrong ones to measure against. The 90% Wilson CI on 8/150 is
[3.0%, 9.2%] (`scipy.stats.binomtest(8, 150).proportion_ci(confidence_level=0.90,
method='wilson')`). Copying 7B's N_EXTRA=200 would only be expected to add
~6-18 more wrong items (point estimate ~11), for a total of ~14-26 --
plausibly still short of the registered minimum of 30.

Planning off the point estimate (not the pessimistic CI bound, which would
require an impractical ~734 extra items and did not match what actually
happened for 7B -- that run's realized rate tracked its point estimate, not
its CI floor): **N_EXTRA=500** gives an expected ~26.7 new wrong items
(8 + 0.053 x 500), for a total of ~34.7 -- a comfortable margin over 30,
similar in proportion to the margin 7B's 200-item draw gave. This is a
plan, not a guarantee: the registered minimum is checked honestly against
whatever the run actually produces, and if it falls short the fix is the
same as it would have been for 7B -- call
`load_fermat_extra_error_items(skip=650, ...)` for a second batch, disjoint
from this one by construction.

**Prerequisite for running:** cells 2-3 (install, auth) must run this
session. Model load is 3B, not 7B -- much lighter, should be fast even on a
T4.


In [1]:
# Install cell: GPU-dependent packages only.
# `datasets` is intentionally NOT pinned/cached across sessions -- see
# pilot/data.py's module docstring for why FERMAT is re-downloaded fresh
# every time rather than persisted to Drive.
%pip install -q transformers accelerate qwen-vl-utils datasets huggingface_hub


In [2]:
# Auth & code/results access cell. Identical to notebook 06's -- reuses the
# HF/GitHub tokens already cached on Drive from prior sessions.
import json
import os
from getpass import getpass

from huggingface_hub import login

from google.colab import drive

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
DRIVE_MODEL_CACHE = f"{PROJECT_DIR}/model_cache"
os.makedirs(DRIVE_MODEL_CACHE, exist_ok=True)

TOKEN_FILE = f"{PROJECT_DIR}/.tokens.json"
RESET_TOKENS = False


def get_token(name, prompt):
    tokens = {}
    if os.path.exists(TOKEN_FILE):
        with open(TOKEN_FILE) as f:
            tokens = json.load(f)
    if RESET_TOKENS or not tokens.get(name):
        tokens[name] = getpass(prompt).strip()
        with open(TOKEN_FILE, "w") as f:
            json.dump(tokens, f)
        os.chmod(TOKEN_FILE, 0o600)
        print(f"Saved {name} to Drive -- you will not be asked for it again.")
    return tokens[name]


HF_TOKEN = get_token("HF_TOKEN", "Hugging Face token (asked once): ")
GH_TOKEN = get_token("GH_TOKEN", "GitHub token with 'repo' scope (asked once): ")

if not HF_TOKEN.startswith("hf_"):
    raise ValueError(
        "Stored Hugging Face token does not start with 'hf_'. Set "
        "RESET_TOKENS = True and re-run this cell to replace it."
    )

login(token=HF_TOKEN)
print("Hugging Face login OK")

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/

import importlib
import sys

REPO_DIR = os.path.abspath("repo")
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
importlib.invalidate_caches()

import pilot.data
import pilot.prompts
import pilot.parsing
import pilot.entropy

print(f"pilot package imported from: {os.path.dirname(pilot.__file__)}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Hugging Face login OK
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for pilot (pyproject.toml) ... done
pilot package imported from: /content/repo/pilot


In [3]:
# Model load cell. 3B, not 7B -- notebook 03's original model (no
# quantization concept in this project's 3B runs; it always loads cleanly in
# bf16, so no OOM/4-bit fallback is implemented here, matching notebook 03).
import torch
from transformers import AutoProcessor, Qwen2_5_VLForConditionalGeneration

MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.bfloat16,
    device_map="auto",
    cache_dir=DRIVE_MODEL_CACHE,
)
processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=DRIVE_MODEL_CACHE)
print(f"Loaded {MODEL_ID}")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Loaded Qwen/Qwen2.5-VL-3B-Instruct


In [4]:
# Extra-items sample cell. Draws ONLY new has_error=1 items, disjoint from
# the 150 the reference 3B run (notebook 03) already used -- same seed=42,
# same skip=150, so this is guaranteed the very next slice in the ordering
# notebook 03's load_fermat_balanced already consumed the front of.
import logging

import pilot.data

logging.basicConfig(level=logging.WARNING, force=True)

SEED = 42
SKIP = 150          # has_error=1 items the reference 3B run already covered
N_EXTRA = 500        # see the intro cell for the sizing rationale

extra_sample = pilot.data.load_fermat_extra_error_items(
    n_extra=N_EXTRA, seed=SEED, skip=SKIP
)
N_EXTRA = len(extra_sample)  # may shrink if the pool ran short -- keep in sync
print(f"{N_EXTRA} additional has_error=1 items drawn (items {SKIP} to {SKIP + N_EXTRA - 1} "
      f"in the seed={SEED} error-item ordering, disjoint from the reference run's first {SKIP})")


500 additional has_error=1 items drawn (items 150 to 649 in the seed=42 error-item ordering, disjoint from the reference run's first 150)


In [5]:
# Grading generation for the extra items. K=5, matching the reference run.
# 3B is small and has never OOM'd in this project, but the batch-backoff
# ladder is kept anyway -- cheap insurance, same pattern as 05/06.
#
# Checkpoint entries use the key name grading_samples_raw (not samples_raw)
# to match the reference checkpoint's schema (notebook 03 predates the
# samples_raw/quantized convention introduced in 04/05) -- this lets the
# merge cell use ONE score_entry function for both checkpoints instead of
# two schema-specific variants.
import gc
import json
import os
import time

import torch
from qwen_vl_utils import process_vision_info
from tqdm.auto import tqdm

import pilot.prompts

K_GRADING = 5
TEMP = 0.7
_BATCH_LADDER = [5, 2, 1]
_batch_state = {"index": 0}

META_FIELDS = ("orig_q", "pert_a", "has_error", "handwriting_style", "image_quality")
INFRA_EXCEPTIONS = (ConnectionError, TimeoutError, torch.cuda.OutOfMemoryError, OSError)


def _generate_batch(messages, n: int, temperature: float):
    text_prompt = processor.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text_prompt], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs, max_new_tokens=512, do_sample=True,
            temperature=temperature, num_return_sequences=n,
        )

    trimmed = output_ids[:, inputs["input_ids"].shape[1]:]
    texts = processor.batch_decode(
        trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
    )
    del output_ids, inputs
    gc.collect()
    torch.cuda.empty_cache()
    return texts


def generate_grading(messages, n: int, temperature: float):
    texts = []
    last_exc = None
    while len(texts) < n:
        want = n - len(texts)
        size = min(_BATCH_LADDER[_batch_state["index"]], want)
        for attempt in range(3):
            try:
                texts += _generate_batch(messages, size, temperature)
                last_exc = None
                break
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if _batch_state["index"] + 1 < len(_BATCH_LADDER):
                    _batch_state["index"] += 1
                    print(f"  OOM at batch {size}; dropping to "
                          f"{_BATCH_LADDER[_batch_state['index']]} for the rest of the run.",
                          flush=True)
                    size = min(_BATCH_LADDER[_batch_state["index"]], n - len(texts))
                    continue
                raise
            except INFRA_EXCEPTIONS as exc:
                last_exc = exc
                gc.collect()
                torch.cuda.empty_cache()
                if attempt < 2:
                    time.sleep(5)
        if last_exc is not None:
            raise last_exc
    return texts


CHECKPOINT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm/checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
model_slug = MODEL_ID.split("/")[-1]
extra_grading_path = (f"{CHECKPOINT_DIR}/grading_3b_extra_error_k{K_GRADING}_{model_slug}"
                      f"_n{N_EXTRA}_skip{SKIP}_seed{SEED}.jsonl")

extra_grading_results = []
if os.path.exists(extra_grading_path):
    with open(extra_grading_path) as f:
        extra_grading_results = [json.loads(line) for line in f if line.strip()]
    valid = []
    for idx, entry in enumerate(extra_grading_results[:N_EXTRA]):
        item = extra_sample[idx]
        if not all(entry["item"].get(k) == item[k] for k in META_FIELDS):
            print(f"Checkpoint item {idx + 1} does not match sample order; resuming there.")
            break
        if len(entry.get("grading_samples_raw", [])) != K_GRADING:
            break
        valid.append(entry)
    if len(valid) != len(extra_grading_results):
        with open(extra_grading_path, "w") as f:
            for e in valid:
                f.write(json.dumps(e, default=str) + "\n")
        print(f"Truncated checkpoint from {len(extra_grading_results)} to {len(valid)} valid items.")
    extra_grading_results = valid
    print(f"Resuming from {len(extra_grading_results)} completed items")

if len(extra_grading_results) >= N_EXTRA:
    print(f"All {N_EXTRA} items already done.")
else:
    print(f"Starting from item {len(extra_grading_results) + 1}/{N_EXTRA} "
          f"({N_EXTRA - len(extra_grading_results)} remaining)", flush=True)
    with tqdm(total=(N_EXTRA - len(extra_grading_results)) * K_GRADING,
              desc="grading (extra, 3B)", unit="sample") as pbar:
        for item_idx, item in enumerate(extra_sample):
            if item_idx < len(extra_grading_results):
                continue
            _t0 = time.time()
            messages = pilot.prompts.build_grading_messages(item["image"])
            texts = generate_grading(messages, K_GRADING, TEMP)
            _elapsed = time.time() - _t0
            entry = {
                "item": {k: item[k] for k in META_FIELDS},
                "grading_samples_raw": texts,
                "model_id": MODEL_ID,
                "elapsed_seconds": _elapsed,
            }
            extra_grading_results.append(entry)
            with open(extra_grading_path, "a") as f:
                f.write(json.dumps(entry, default=str) + "\n")
                f.flush()
            pbar.update(K_GRADING)
            print(f"  item {item_idx + 1}/{N_EXTRA}: {_elapsed:.1f}s "
                  f"({len(extra_grading_results)}/{N_EXTRA} done)", flush=True)

print(f"extra_grading_results: {len(extra_grading_results)} items")


Resuming from 270 completed items
Starting from item 271/500 (230 remaining)


grading (extra, 3B):   0%|          | 0/1150 [00:00<?, ?sample/s]

  item 271/500: 18.5s (271/500 done)
  item 272/500: 11.4s (272/500 done)
  item 273/500: 19.9s (273/500 done)
  item 274/500: 12.8s (274/500 done)
  item 275/500: 10.3s (275/500 done)
  item 276/500: 11.5s (276/500 done)
  item 277/500: 16.9s (277/500 done)
  item 278/500: 15.8s (278/500 done)
  item 279/500: 9.6s (279/500 done)
  item 280/500: 12.7s (280/500 done)
  item 281/500: 15.7s (281/500 done)
  item 282/500: 11.7s (282/500 done)
  item 283/500: 14.5s (283/500 done)
  item 284/500: 11.4s (284/500 done)
  item 285/500: 15.4s (285/500 done)
  item 286/500: 16.9s (286/500 done)
  item 287/500: 14.9s (287/500 done)
  item 288/500: 20.7s (288/500 done)
  item 289/500: 13.0s (289/500 done)
  item 290/500: 17.0s (290/500 done)
  item 291/500: 15.9s (291/500 done)
  item 292/500: 14.7s (292/500 done)
  item 293/500: 28.5s (293/500 done)
  item 294/500: 16.7s (294/500 done)
  item 295/500: 11.3s (295/500 done)
  item 296/500: 11.2s (296/500 done)
  item 297/500: 9.1s (297/500 done)
  i

In [6]:
# Merge with the 3B reference run (notebook 03) and re-run the stratified
# analysis. Reads the REFERENCE CHECKPOINT (raw text) directly, not the
# already-scored CSV -- same reasoning as notebook 06.
#
# Schema note: the reference checkpoint predates the samples_raw/quantized
# convention from 04/05 -- its entries have transcription_samples_raw AND
# grading_samples_raw (this notebook only needs the latter), and no
# `quantized` field at all (notebook 03's 3B model load never quantizes).
# The only cross-checkpoint consistency check that applies here is
# model_id, not quantization.
import importlib
import json
import os

import pandas as pd

import pilot.entropy
import pilot.parsing
import pilot.plotting

for m in (pilot.parsing, pilot.entropy, pilot.plotting):
    importlib.reload(m)

REFERENCE_CHECKPOINT = (
    f"{CHECKPOINT_DIR}/scaleup_{model_slug}_n300_seed{SEED}_bal50_kt5_kg5.jsonl"
)
if not os.path.exists(REFERENCE_CHECKPOINT):
    raise AssertionError(
        f"No reference checkpoint found at {REFERENCE_CHECKPOINT}. It is the "
        "raw 2026-08-02 n=300 3B scaleup run (notebook 03); without it there "
        "is nothing to merge the extra items into."
    )

with open(REFERENCE_CHECKPOINT) as f:
    reference_entries = [json.loads(line) for line in f if line.strip()]
assert len(reference_entries) == 300, (
    f"Expected 300 reference items, found {len(reference_entries)} -- the "
    "checkpoint may be incomplete."
)


def score_entry(entry, model_id_fallback):
    """Score one raw checkpoint entry (grading only), keeping raw text and
    per-sample digits alongside the usual derived columns. Works for both
    checkpoint schemas: the reference entry has no model_id field of its own
    (notebook 03 didn't record it per-entry, only once at the notebook
    level), so model_id_fallback covers that case."""
    digits = [pilot.parsing.parse_grading(t) for t in entry["grading_samples_raw"]]
    labels = [None if d is None else str(d) for d in digits]
    majority, _ = pilot.entropy.majority_cluster(labels)
    said_error = majority == "1"
    return {
        "orig_q": entry["item"]["orig_q"],
        "pert_a": entry["item"]["pert_a"],
        "has_error": entry["item"]["has_error"],
        "grading_correct": majority in {"0", "1"} and said_error == bool(entry["item"]["has_error"]),
        "said_error": said_error,
        "reasoning_entropy": pilot.entropy.cluster_entropy(labels),
        "n_grading_parse_failures": sum(1 for d in digits if d is None),
        "majority_digit": majority,
        "parsed_digits": digits,
        "all_grading_samples_raw": entry["grading_samples_raw"],
        "model_id": entry.get("model_id", model_id_fallback),
    }


reference_df = pd.DataFrame(score_entry(e, MODEL_ID) for e in reference_entries)
extra_df = pd.DataFrame(score_entry(e, MODEL_ID) for e in extra_grading_results)

# model_id consistency check -- the relevant cross-checkpoint guard here,
# since 3B has no quantization concept to mismatch on.
ref_model_ids = set(reference_df["model_id"].unique())
extra_model_ids = set(extra_df["model_id"].unique())
if ref_model_ids != extra_model_ids or ref_model_ids != {MODEL_ID}:
    raise RuntimeError(
        f"model_id mismatch: reference={ref_model_ids}, extra={extra_model_ids}, "
        f"this session={MODEL_ID}. Refusing to merge -- these may not be "
        "measurements of the same model."
    )
print(f"model_id check OK: both runs are {MODEL_ID}")

# Disjointness check, not just an assumption.
overlap = set(zip(reference_df["orig_q"], reference_df["pert_a"])) & \
          set(zip(extra_df["orig_q"], extra_df["pert_a"]))
assert not overlap, f"{len(overlap)} items overlap between reference and extra -- sampling bug"

combined = pd.concat([reference_df, extra_df], ignore_index=True)
print(f"Combined: {len(reference_df)} reference + {len(extra_df)} extra = {len(combined)} total")

gt = combined["has_error"].astype(bool)
print(f"  has_error=1: {int(gt.sum())} items, {int((~combined.loc[gt,'grading_correct']).sum())} misgraded")
print(f"  has_error=0: {int((~gt).sum())} items, {int((~combined.loc[~gt,'grading_correct']).sum())} misgraded")

print()
print("=" * 70)
print("STRATIFIED ANALYSIS (combined, 3B)")
print("=" * 70)
out = pilot.plotting.stratified_auroc(
    combined, "reasoning_entropy", "grading_correct", "has_error", n_boot=10000, seed=0
)
for level, s in out["strata"].items():
    minority = min(s["n_error"], s["n_correct"])
    powered = minority >= pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
    print(f"  has_error={level}  n={s['n_items']:3d}  n_wrong={s['n_error']:3d}  "
          f"AUROC {s['auroc']:.3f} [{s['ci_low']:.3f}, {s['ci_high']:.3f}]  "
          f"minority={minority}  {'POWERED' if powered else 'still underpowered'}")
print(f"  sign_reversal      : {out['sign_reversal']}")
print(f"  pooled_understates : {out['pooled_understates']}")

print()
print("=" * 70)
print("VERDICT (reusing the registered 0.70 threshold, not a new one)")
print("=" * 70)
error_stratum = out["strata"][True]
minority = min(error_stratum["n_error"], error_stratum["n_correct"])
min_n = pilot.plotting.SCALEUP_PREREGISTRATION["min_minority_class"]
threshold = pilot.plotting.SCALEUP_PREREGISTRATION["reasoning_stratum_auroc_min"]

if minority < min_n:
    print(f"  Still underpowered ({minority} < {min_n}). N_EXTRA was not enough; "
          f"draw more with load_fermat_extra_error_items(skip={SKIP + N_EXTRA}, ...) "
          "to extend further.")
elif error_stratum["auroc"] >= threshold and error_stratum["excludes_chance"]:
    print(f"  CONFIRMED: AUROC {error_stratum['auroc']:.3f} clears the registered "
          f"{threshold} threshold with adequate power ({minority} >= {min_n}).")
else:
    print(f"  NOT CONFIRMED: adequately powered ({minority} >= {min_n}) but AUROC "
          f"{error_stratum['auroc']:.3f} does not clear {threshold}, or its CI includes chance.")

print()
print("=" * 70)
print("MODEL-SIZE COMPARISON (this is the whole point of this notebook)")
print("=" * 70)
print(f"  3B has_error=1, powered : {error_stratum['auroc']:.3f} "
      f"[{error_stratum['ci_low']:.3f}, {error_stratum['ci_high']:.3f}]  n_wrong={error_stratum['n_error']}")
print(f"  7B has_error=1, powered : 0.834 [0.768, 0.891]  n_wrong=38  (report S7.4)")
print("  If these land close together (as 3B's and 7B's underpowered point")
print("  estimates already did, 0.836 vs 0.761), the stratified effect is")
print("  model-size-independent. If 3B's confirmed value is much lower, 7B's")
print("  result does not generalize down to a smaller model.")


model_id check OK: both runs are Qwen/Qwen2.5-VL-3B-Instruct
Combined: 300 reference + 500 extra = 800 total
  has_error=1: 650 items, 42 misgraded
  has_error=0: 150 items, 137 misgraded

STRATIFIED ANALYSIS (combined, 3B)
  has_error=False  n=150  n_wrong=137  AUROC 0.239 [0.128, 0.374]  minority=13  still underpowered
  has_error=True  n=650  n_wrong= 42  AUROC 0.854 [0.796, 0.902]  minority=42  POWERED
  sign_reversal      : True
  pooled_understates : True

VERDICT (reusing the registered 0.70 threshold, not a new one)
  CONFIRMED: AUROC 0.854 clears the registered 0.7 threshold with adequate power (42 >= 30).

MODEL-SIZE COMPARISON (this is the whole point of this notebook)
  3B has_error=1, powered : 0.854 [0.796, 0.902]  n_wrong=42
  7B has_error=1, powered : 0.834 [0.768, 0.891]  n_wrong=38  (report S7.4)
  If these land close together (as 3B's and 7B's underpowered point
  estimates already did, 0.836 vs 0.761), the stratified effect is
  model-size-independent. If 3B's confi

In [7]:
# Save cell: Drive first, then repo + push. Distinctly named -- never
# overwrites the 3B reference results file or the 7B stratum-powered one.
import time

TIMESTAMP = time.strftime("%Y%m%dT%H%M%SZ", time.gmtime())
model_slug_lower = MODEL_ID.split("/")[-1].lower().replace("_", "-")
OUT_NAME = f"grading_3b_stratum_powered_n{len(combined)}_{model_slug_lower}_{TIMESTAMP}.csv"

DRIVE_RESULTS_DIR = f"{PROJECT_DIR}/results"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
drive_out_path = f"{DRIVE_RESULTS_DIR}/{OUT_NAME}"
combined.to_csv(drive_out_path, index=False)
print(f"Backup written to {drive_out_path}")

repo_out_path = f"repo/results/{OUT_NAME}"
os.makedirs("repo/results", exist_ok=True)
combined.to_csv(repo_out_path, index=False)
print(f"Wrote {repo_out_path} ({len(combined)} rows)")

import subprocess

GH_REMOTE = f"https://{GH_TOKEN}@github.com/sepehrmaleki369/uncertainty-math-vlm.git"


def git(*args):
    result = subprocess.run(["git", "-C", "repo"] + list(args), capture_output=True, text=True)
    # Redact the token from any echoed command/output before printing.
    out = (result.stdout + result.stderr).replace(GH_TOKEN, "***")
    print(out)
    return result.returncode


# A fresh Colab runtime has no git user.name/email configured, which fails
# the commit below with "Author identity unknown" -- a different, earlier
# failure than the usual 403 push rejection, and one this notebook actually
# hit on its first real run. Configuring identity here doesn't guarantee
# the push succeeds, but it stops this specific failure mode.
git("config", "user.email", "colab-pilot@localhost")
git("config", "user.name", "Colab Pilot Run")
git("add", f"results/{OUT_NAME}")
git("commit", "-m", f"Add 3B has_error=1 stratum power extension ({OUT_NAME})")
git("fetch", "origin", "main")
rc = git("rebase", "origin/main")
if rc != 0:
    print("Rebase failed -- resolve manually. The CSV is safe on Drive regardless.")
else:
    push_rc = subprocess.run(
        ["git", "-C", "repo", "push", GH_REMOTE, "HEAD:main"],
        capture_output=True, text=True,
    )
    out = (push_rc.stdout + push_rc.stderr).replace(GH_TOKEN, "***")
    print(out)
    if push_rc.returncode != 0:
        print("Push failed (see above). The CSV is safe on Drive and in "
              "repo/results/ -- retry the push without re-running the model.")


Backup written to /content/drive/MyDrive/uncertainty-math-vlm/results/grading_3b_stratum_powered_n800_qwen2.5-vl-3b-instruct_20260806T150044Z.csv
Wrote repo/results/grading_3b_stratum_powered_n800_qwen2.5-vl-3b-instruct_20260806T150044Z.csv (800 rows)

Author identity unknown

*** Please tell me who you are.

Run

  git config --global user.email "you@example.com"
  git config --global user.name "Your Name"

to set your account's default identity.
Omit --global to set the identity only in this repository.

fatal: unable to auto-detect email address (got 'root@2d978a24c4d6.(none)')

From https://github.com/sepehrmaleki369/uncertainty-math-vlm
 * branch            main       -> FETCH_HEAD

error: cannot rebase: Your index contains uncommitted changes.
error: Please commit or stash them.

Rebase failed -- resolve manually. The CSV is safe on Drive regardless.
